In [1]:
import pandas as pd
import numpy as np

In [2]:
# Read input data from FRBNY website
bra_gdp =  pd.read_excel("inputData/bra_data.xlsx", sheet_name="GDP")
bra_gdp = bra_gdp.set_index("Date")
bra_gdp['gdp.log'] = np.log(bra_gdp['GDP'])
#drop gdp raw 
bra_gdp = bra_gdp.drop(columns = ['GDP'])
#convert index to datetime(e.g. 3/31/2025)
bra_gdp.index = pd.to_datetime(bra_gdp.index,)

bra_gdp = bra_gdp.resample("QS").mean()
bra_gdp





,gdp.log
Date,
1996-01-01,25.423631
1996-04-01,25.454440
1996-07-01,25.484277
1996-10-01,25.533424
1997-01-01,25.510705
...,...
2022-07-01,26.087128
2022-10-01,26.089261
2023-01-01,26.099137


In [3]:
bra_inflation =  pd.read_csv("inputData/bra_infl.csv", sep=";")
bra_inflation = bra_inflation.set_index("Date")
#set first row to 100
bra_inflation['index'] = np.nan
bra_inflation['index'].iloc[0] = 100

for i in range(1,len(bra_inflation)):
    bra_inflation['index'][i] = ((bra_inflation['inflation'][i]/100) * bra_inflation['index'][i-1]) + bra_inflation['index'][i-1]

#convert 1996/01 to 1996-01-01
bra_inflation.index = pd.to_datetime(bra_inflation.index, format='%m/%Y')

#drop index column 
bra_inflation = bra_inflation.drop(columns = ['inflation'])

#resample 'index' column to quarterly
bra_inflation = bra_inflation.resample('QS').mean()
bra_inflation['growth'] = bra_inflation.pct_change() * 100
pd.DataFrame(bra_inflation)
bra_inflation = bra_inflation.dropna()

bra_inflation


# # #annualize quarterly growth
bra_inflation['annualized'] =  100 * ((1 + bra_inflation['growth']/100)**4 - 1)
bra_inflation['annualized']
bra_inflation_final = bra_inflation['annualized'].to_frame()
bra_inflation_final['expectations'] = bra_inflation_final.rolling(4).mean()
bra_inflation_final = bra_inflation_final.dropna()

bra_inflation_final

C:\Users\tfarotimi\AppData\Local\Temp\ipykernel_29756\4118426606.py:5: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  bra_inflation['index'].iloc[0] = 100
C:\Users\tfarotimi\AppData\Local\Temp\ipykernel_29756\4118426606.py:8: FutureWarning: S

,annualized,expectations
Date,,
1997-01-01,7.655992,9.098441
1997-04-01,6.516911,7.286217
1997-07-01,4.502909,6.200688
1997-10-01,4.393631,5.767361
1998-01-01,4.445942,4.964849
...,...,...
2024-01-01,4.365005,3.620191
2024-04-01,3.659633,3.561626
2024-07-01,3.714693,3.815573


In [4]:
#.interest    
bra_interest =  pd.read_csv("inputData/bra_interest.csv", sep=";")
bra_interest = bra_interest.set_index("Date")
#convert 07/04/2024 to datetime .index 
bra_interest.index = pd.to_datetime(bra_interest.index, format='%d/%m/%Y')
bra_interest = bra_interest.resample('QS').mean()
bra_interest.columns = ['interest']
bra_interest = bra_interest.dropna()
bra_interest

,interest
Date,
1999-01-01,44.222222
1999-04-01,28.769231
1999-07-01,19.913043
1999-10-01,19.000000
2000-01-01,18.983516
...,...
2024-04-01,10.604396
2024-07-01,10.532609
2024-10-01,11.266304


In [5]:
#merge bra_gdp, bra_inflation_final, bra_interest
bra_data = pd.concat([bra_gdp, bra_inflation_final, bra_interest], axis=1).dropna()
bra_data

,gdp.log,annualized,expectations,interest
Date,,,,
1999-01-01,25.496316,5.452907,2.718339,44.222222
1999-04-01,25.495037,5.426667,3.218620,28.769231
1999-07-01,25.504927,4.364636,3.984807,19.913043
1999-10-01,25.521586,5.173614,5.104456,19.000000
2000-01-01,25.514418,4.707844,4.918190,18.983516
...,...,...,...,...
2022-07-01,26.087128,5.205470,7.047116,13.565217
2022-10-01,26.089261,3.220069,6.429737,13.750000
2023-01-01,26.099137,3.632665,5.522850,13.750000


In [6]:
stringency = pd.read_csv("inputData/stringency.csv")
stringency_data = stringency[['CountryCode', 'Date', 'StringencyIndex_Average']]
stringency_data = stringency_data[stringency_data['CountryCode'] == 'BRA']
stringency_data = stringency_data.drop(columns=['CountryCode'])
# stringency_data['Date'] = pd.to_datetime(stringency_data['Date'])
stringency_data = stringency_data.set_index('Date')
#process 20200101 to 2020-01-01
stringency_data.index = pd.to_datetime(stringency_data.index, format='%Y%m%d')
stringency_data = stringency_data.resample('QS').mean()
stringency_data = stringency_data.rename(columns={'StringencyIndex_Average': 'covid.indicator'})
stringency_data.index = stringency_data.index.to_period('Q')
#add rows up to 2024Q1 with stringency declining linearly to 0
stringency_data = stringency_data.reindex(pd.period_range(start='2020Q1', end='2024Q1', freq='Q', name='Date'))
stringency_data.loc['2024Q1'] = 0
stringency_data = stringency_data.interpolate(method='linear')
#add rows forom 1990Q1 to 2019Q4 with stringency = 0
stringency_data = stringency_data.reindex(pd.period_range(start='1990Q1', end='2024Q1', freq='Q', name='Date'))
stringency_data = stringency_data.fillna(0)
stringency_data = stringency_data.dropna()
stringency_data

,covid.indicator
Date,
1990Q1,0.000
1990Q2,0.000
1990Q3,0.000
1990Q4,0.000
1991Q1,0.000
...,...
2023Q1,17.776
2023Q2,13.332
2023Q3,8.888


In [7]:
#bra index to quarter 
bra_data.index = bra_data.index.to_period('Q')
#merge bra_data and stringency_data
bra_data = pd.merge(bra_data, stringency_data, left_index=True, right_index=True, how='outer')
bra_data = bra_data.dropna()
bra_data

,gdp.log,annualized,expectations,interest,covid.indicator
Date,,,,,
1999Q1,25.496316,5.452907,2.718339,44.222222,0.000000
1999Q2,25.495037,5.426667,3.218620,28.769231,0.000000
1999Q3,25.504927,4.364636,3.984807,19.913043,0.000000
1999Q4,25.521586,5.173614,5.104456,19.000000,0.000000
2000Q1,25.514418,4.707844,4.918190,18.983516,0.000000
...,...,...,...,...,...
2022Q3,26.087128,5.205470,7.047116,13.565217,24.421739
2022Q4,26.089261,3.220069,6.429737,13.750000,22.220000
2023Q1,26.099137,3.632665,5.522850,13.750000,17.776000


In [8]:
#rename braumns
bra_data.columns = ['gdp.log', 'inflation', 'inflation.expectations', 'interest', 'covid.indicator']

#add braumns of 0 called covid.indicator
#bra_data['covid.indicator'] = 0

#save data to excel
bra_data.to_excel("inputData/Holston_Laubach_Williams_BRA.xlsx", sheet_name="BRA Input Data" )
